In [2]:
import pandas as pd
from sqlalchemy import create_engine

# --- ANDMEBAASI ÜHENDUS ---
DB_USER = "postgres"
DB_PASSWORD = "andmed"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "postgres"

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

print("Andmete laadimine käib...")

# Tabelite sisse lugemine
dim_date = pd.read_sql("SELECT * FROM dim_date;", engine)
dim_products = pd.read_sql("SELECT * FROM dim_products;", engine)
dim_customers = pd.read_sql("SELECT * FROM dim_customers;", engine)
fact_order_items_clean = pd.read_sql("SELECT * FROM fact_order_items_clean;", engine)
fact_cart_items = pd.read_sql("SELECT * FROM fact_cart_items;", engine)

print(" Kõik tabelid on edukalt laetud!")

Andmete laadimine käib...
 Kõik tabelid on edukalt laetud!


## Customer_type sataatus: guest/registered ##

In [3]:
# Vaatame, millised klienditüübid veerus 'customer_type' üldse eksisteerivad ja kui palju neid on
print("--- Klienditüübid (customer_type) ---")
print(fact_order_items_clean['customer_type'].value_counts(dropna=False))

# Kontrollime, kas meil on üldse kliente erinevat tüüpi tellimustega
client_type_variations = fact_order_items_clean.groupby('customer_id')['customer_type'].nunique()
mixed_count = (client_type_variations > 1).sum()

print(f"\nMitme erineva tüübiga klientide arv: {mixed_count}")

--- Klienditüübid (customer_type) ---
customer_type
guest         9882
registered    6045
Name: count, dtype: int64

Mitme erineva tüübiga klientide arv: 161


### Näiteks ###

In [4]:
# Lubame kuvada kõik read ilma kärpimiseta
pd.set_option('display.max_rows', None)

# Filtreerime ainult täidetud tellimused (status == 'complete')
completed_orders = fact_order_items_clean[fact_order_items_clean['status'] == 'complete']

# Otsime kliente, kellel 'customer_type' muutus ja kellel on rohkem kui 5 tellimust
mixed_clients_5plus = completed_orders.groupby('customer_id').filter(
    lambda x: (x['customer_type'].nunique() > 1) and (x['order_id'].nunique() > 5)
)

if not mixed_clients_5plus.empty:
    # Võtame esimese leitud kliendi ID
    sample_id = mixed_clients_5plus['customer_id'].iloc[0]
    sample_history = completed_orders[completed_orders['customer_id'] == sample_id]
    
    print(f"--- Kliendi (ID: {sample_id}) KÕIK ostud ({sample_history['order_id'].nunique()} tk) ---")
    
    # Kuvame kõik ostud kronoloogilises järjekorras
    display(
        sample_history[['order_id', 'customer_id', 'date_id', 'status', 'customer_type', 'revenue_with_vat']]
        .drop_duplicates()
        .sort_values(by='date_id')
    )
else:
    print("Klienti, kellel oleks nii muutuv staatus kui ka üle 5 ostu, ei leitud. Vaatame lihtsalt esimest muutuva staatusega klienti:")
    mixed_completed_clients = completed_orders.groupby('customer_id').filter(lambda x: x['customer_type'].nunique() > 1)
    sample_id = mixed_completed_clients['customer_id'].iloc[0]
    sample_history = completed_orders[completed_orders['customer_id'] == sample_id]
    display(
        sample_history[['order_id', 'customer_id', 'date_id', 'status', 'customer_type', 'revenue_with_vat']]
        .drop_duplicates()
        .sort_values(by='date_id')
    )

--- Kliendi (ID: C02052) KÕIK ostud (7 tk) ---


,order_id,customer_id,date_id,status,customer_type,revenue_with_vat
1329,000000499,C02052,20220605,complete,registered,44.0
1611,000000605,C02052,20220907,complete,guest,33.0
1738,000000670,C02052,20221031,complete,registered,36.9
1898,000000741,C02052,20221222,complete,registered,49.2
2136,000000840,C02052,20230226,complete,registered,54.8
2365,000000948,C02052,20230504,complete,registered,82.2
2796,000001192,C02052,20230821,complete,registered,82.2


## 1. Müügidünaamika ##

### 1.1 Käibe ja käibe kasvumäära dünaamika 2021–2025 ###

In [5]:
# 0. Andmete ettevalmistus: ühendame fakstitabeli kuupäevade tabeliga
df = fact_order_items_clean.merge(dim_date[['date_id', 'year', 'm', 'day']], on='date_id', how='left')

# Filtreerime välja ainult edukad tellimused (status == 'complete')
df_complete = df[df['status'] == 'complete'].copy()

# --- OSA 1: Müügidünaamika täisaastate lõikes (netokäive, ümardatud täisarvudeni) ---
kaive_aastas = df_complete.groupby('year')['revenue_without_vat'].sum().reset_index(name='Käive_EUR')
kaive_aastas['Käive_EUR'] = kaive_aastas['Käive_EUR'].round(0) # Ümardamine täisarvuks
kaive_aastas = kaive_aastas.sort_values('year').reset_index(drop=True)

# Arvutame aastase kasvumäära (%) võrreldes eelneva aastaga
kaive_aastas['Kasv_%'] = kaive_aastas['Käive_EUR'].pct_change() * 100
kaive_aastas['Kasv_%'] = kaive_aastas['Kasv_%'].fillna(0).round(1)

# Arvutame üldise kogusumma (Kokku)
kokku_summa = round(kaive_aastas['Käive_EUR'].sum(), 0)

# Lisame "Kokku" rea DataFrame'i lõppu
kokku_rida = pd.DataFrame({
    'year': ['Kokku'],
    'Käive_EUR': [kokku_summa],
    'Kasv_%': [None]
})
kaive_aastas_kokku = pd.concat([kaive_aastas, kokku_rida], ignore_index=True)

# Vormindame esitluse jaoks (täisarvud tuhandete eraldajaga)
vaatamise_tabel = kaive_aastas_kokku.copy()
vaatamise_tabel['Aasta'] = vaatamise_tabel['year'].astype(str)
vaatamise_tabel['Netokäive (EUR)'] = vaatamise_tabel['Käive_EUR'].apply(lambda x: f"{int(x):,} €".replace(",", " ") if pd.notnull(x) else "")
vaatamise_tabel['Kasvumäär (%)'] = vaatamise_tabel['Kasv_%'].apply(lambda x: f"+{x}%" if pd.notnull(x) and x > 0 else (f"{x}%" if pd.notnull(x) else "-"))

print("=== 1. MÜÜGIDÜNAAMIKA TÄISAASTATE LÕIKES (NETOKÄIVE, TÄISARVUD) ===")
display(vaatamise_tabel[['Aasta', 'Netokäive (EUR)', 'Kasvumäär (%)']])


# --- OSA 2: Perioodide võrdlus (01.01 - 17.09) 2025 vs 2026 (täisarvud) ---
periood_2025 = round(df_complete[
    (df_complete['year'] == 2025) & 
    ((df_complete['m'] < 9) | ((df_complete['m'] == 9) & (df_complete['day'] <= 17)))
]['revenue_without_vat'].sum(), 0)

periood_2026 = round(df_complete[
    (df_complete['year'] == 2026) & 
    ((df_complete['m'] < 9) | ((df_complete['m'] == 9) & (df_complete['day'] <= 17)))
]['revenue_without_vat'].sum(), 0)

muutus_eur = periood_2026 - periood_2025
muutus_protsent = (muutus_eur / periood_2025) * 100

muutus_protsent_str = f"+{muutus_protsent:.1f}%" if muutus_protsent > 0 else f"{muutus_protsent:.1f}%"

print("\n=== 2. VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) - NETOKÄIVE ===")
vertlus_tabel = pd.DataFrame({
    'Vaadeldav periood': ['01.01 - 17.09.2025', '01.01 - 17.09.2026', 'Muutus / Kasvumäär'],
    'Netokäive kokku': [
        f"{int(periood_2025):,} €".replace(",", " "), 
        f"{int(periood_2026):,} €".replace(",", " "), 
        muutus_protsent_str
    ]
})
display(vertlus_tabel)
print(f"Erinevus eurodes: {int(muutus_eur):,} €".replace(",", " "))

=== 1. MÜÜGIDÜNAAMIKA TÄISAASTATE LÕIKES (NETOKÄIVE, TÄISARVUD) ===


,Aasta,Netokäive (EUR),Kasvumäär (%)
0,2020,32 €,0.0%
1,2021,2 862 €,+8843.8%
2,2022,12 552 €,+338.6%
3,2023,12 227 €,-2.6%
4,2024,12 028 €,-1.6%
5,2025,26 578 €,+121.0%
6,2026,48 455 €,+82.3%
7,Kokku,114 734 €,-



=== 2. VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) - NETOKÄIVE ===


,Vaadeldav periood,Netokäive kokku
0,01.01 - 17.09.2025,10 891 €
1,01.01 - 17.09.2026,48 455 €
2,Muutus / Kasvumäär,+344.9%


Erinevus eurodes: 37 564 €


### 1.2, 1.3 Täidetud ja tühistatud tellimuste dünaamika ning tühistamiste osakaal ###

In [6]:
# 0. Andmete ettevalmistus: ühendame faktitabeli kuupäevade tabeliga
df = fact_order_items_clean.merge(
    dim_date[['date_id', 'year', 'm', 'day']], on='date_id', how='left'
)

# Teeme staatuse väikesteks tähtedeks ja eemaldame liigsed tühikud
df['status_clean'] = df['status'].astype(str).str.strip().str.lower()


# ==========================================
# OSA 1: Täidetud ja tühistatud tellimuste dünaamika (täisaastad)
# ==========================================
tellimuste_kokkuvote = df.groupby(['year', 'status_clean'])['order_id'].nunique().unstack(fill_value=0)

tulemus_df = pd.DataFrame()
tulemus_df['year'] = tellimuste_kokkuvote.index
tulemus_df['Taidetud'] = tellimuste_kokkuvote.get('complete', 0).values
tulemus_df['Tuhistatud'] = tellimuste_kokkuvote.get('canceled', 0).values
tulemus_df['Kokku'] = tulemus_df['Taidetud'] + tulemus_df['Tuhistatud']

# Ümardame protsendid täisarvudeni
tulemus_df['Tuhistamiste_osa_%'] = (tulemus_df['Tuhistatud'] / tulemus_df['Kokku'] * 100).round(0)
tulemus_df['Kasv_%'] = tulemus_df['Taidetud'].pct_change() * 100
tulemus_df['Kasv_%'] = tulemus_df['Kasv_%'].fillna(0).round(0)

tulemus_df = tulemus_df.sort_values('year').reset_index(drop=True)

# Kokku rida
kokku_taidetud = tulemus_df['Taidetud'].sum()
kokku_tuhistatud = tulemus_df['Tuhistatud'].sum()
kokku_kokku = tulemus_df['Kokku'].sum()
keskmine_tuhistamise_osa = round((kokku_tuhistatud / kokku_kokku) * 100, 0) if kokku_kokku > 0 else 0

kokku_rida = pd.DataFrame({
    'year': ['Kokku'],
    'Taidetud': [kokku_taidetud],
    'Tuhistatud': [kokku_tuhistatud],
    'Kokku': [kokku_kokku],
    'Tuhistamiste_osa_%': [keskmine_tuhistamise_osa],
    'Kasv_%': [None]
})
tulemus_kokku = pd.concat([tulemus_df, kokku_rida], ignore_index=True)

vaatamise_tabel = pd.DataFrame()
vaatamise_tabel['Aasta'] = tulemus_kokku['year'].astype(str)
vaatamise_tabel['Täidetud tellimused'] = tulemus_kokku['Taidetud'].apply(lambda x: f"{int(x):,}".replace(",", " ") if pd.notnull(x) else "")
vaatamise_tabel['Tühistatud tellimused'] = tulemus_kokku['Tuhistatud'].apply(lambda x: f"{int(x):,}".replace(",", " ") if pd.notnull(x) else "")
vaatamise_tabel['Kokku tellimusi'] = tulemus_kokku['Kokku'].apply(lambda x: f"{int(x):,}".replace(",", " ") if pd.notnull(x) else "")

vaatamise_tabel['Tühistamiste osakaal (%)'] = tulemus_kokku['Tuhistamiste_osa_%'].apply(lambda x: f"{int(x)}%" if pd.notnull(x) else "-")
vaatamise_tabel['Täidetud tellimuste kasv (%)'] = tulemus_kokku['Kasv_%'].apply(
    lambda x: f"+{int(x)}%" if pd.notnull(x) and x > 0 else (f"{int(x)}%" if pd.notnull(x) and x < 0 else ("0%" if pd.notnull(x) else "-"))
)

print("=== 1. TÄIDETUD JA TÜHISTATUD TELLIMUSTE DÜNAAMIKA ===")
display(vaatamise_tabel)


# ==========================================
# OSA 2: Võrdlus sama perioodiga (01.01 - 17.09) 2025 vs 2026
# ==========================================
perioodi_filter = (
    ((df['year'] == 2025) | (df['year'] == 2026)) & 
    ((df['m'] < 9) | ((df['m'] == 9) & (df['day'] <= 17)))
)
df_periood = df[perioodi_filter].copy()

periood_kokkuvote = df_periood.groupby(['year', 'status_clean'])['order_id'].nunique().unstack(fill_value=0)

vordlus_df = pd.DataFrame()
vordlus_df['year'] = [2025, 2026]
vordlus_df['Taidetud'] = [
    periood_kokkuvote.loc[2025, 'complete'] if 2025 in periood_kokkuvote.index and 'complete' in periood_kokkuvote.columns else 0,
    periood_kokkuvote.loc[2026, 'complete'] if 2026 in periood_kokkuvote.index and 'complete' in periood_kokkuvote.columns else 0
]
vordlus_df['Tuhistatud'] = [
    periood_kokkuvote.loc[2025, 'canceled'] if 2025 in periood_kokkuvote.index and 'canceled' in periood_kokkuvote.columns else 0,
    periood_kokkuvote.loc[2026, 'canceled'] if 2026 in periood_kokkuvote.index and 'canceled' in periood_kokkuvote.columns else 0
]
vordlus_df['Kokku'] = vordlus_df['Taidetud'] + vordlus_df['Tuhistatud']
vordlus_df['Tuhistamiste_osa_%'] = (vordlus_df['Tuhistatud'] / vordlus_df['Kokku'] * 100).round(0)

taidetud_2025 = vordlus_df.loc[vordlus_df['year'] == 2025, 'Taidetud'].values[0]
taidetud_2026 = vordlus_df.loc[vordlus_df['year'] == 2026, 'Taidetud'].values[0]
muutus_tellimustes = taidetud_2026 - taidetud_2025
kasv_protsent = round((muutus_tellimustes / taidetud_2025) * 100, 0) if taidetud_2025 > 0 else 0

tuhistatud_2025 = vordlus_df.loc[vordlus_df['year'] == 2025, 'Tuhistatud'].values[0]
tuhistatud_2026 = vordlus_df.loc[vordlus_df['year'] == 2026, 'Tuhistatud'].values[0]
muutus_tuhistatud = tuhistatud_2026 - tuhistatud_2025

kokku_2025 = vordlus_df.loc[vordlus_df['year'] == 2025, 'Kokku'].values[0]
kokku_2026 = vordlus_df.loc[vordlus_df['year'] == 2026, 'Kokku'].values[0]
muutus_kokku = kokku_2026 - kokku_2025

muutus_osa_2025 = vordlus_df.loc[vordlus_df['year'] == 2025, 'Tuhistamiste_osa_%'].values[0]
muutus_osa_2026 = vordlus_df.loc[vordlus_df['year'] == 2026, 'Tuhistamiste_osa_%'].values[0]
muutus_osa = muutus_osa_2026 - muutus_osa_2025

muutuse_rida = pd.DataFrame({
    'year': ['Muutus'],
    'Taidetud': [muutus_tellimustes],
    'Tuhistatud': [muutus_tuhistatud],
    'Kokku': [muutus_kokku],
    'Tuhistamiste_osa_%': [muutus_osa]
})

vordlus_kokku = pd.concat([vordlus_df, muutuse_rida], ignore_index=True)

vaatamise_vordlus = pd.DataFrame()
vaatamise_vordlus['Periood / Aasta'] = ['01.01 - 17.09.2025', '01.01 - 17.09.2026', 'Muutus (2026 vs 2025)']

def format_val(val, is_muutus=False):
    if pd.isnull(val):
        return ""
    v = int(val)
    if is_muutus and v > 0:
        return f"+{v:,}".replace(",", " ")
    return f"{v:,}".replace(",", " ")

vaatamise_vordlus['Täidetud tellimused'] = [
    format_val(vordlus_kokku.loc[0, 'Taidetud']),
    format_val(vordlus_kokku.loc[1, 'Taidetud']),
    format_val(vordlus_kokku.loc[2, 'Taidetud'], is_muutus=True)
]
vaatamise_vordlus['Tühistatud tellimused'] = [
    format_val(vordlus_kokku.loc[0, 'Tuhistatud']),
    format_val(vordlus_kokku.loc[1, 'Tuhistatud']),
    format_val(vordlus_kokku.loc[2, 'Tuhistatud'], is_muutus=True)
]
vaatamise_vordlus['Kokku tellimusi'] = [
    format_val(vordlus_kokku.loc[0, 'Kokku']),
    format_val(vordlus_kokku.loc[1, 'Kokku']),
    format_val(vordlus_kokku.loc[2, 'Kokku'], is_muutus=True)
]

vaatamise_vordlus['Tühistamiste osakaal (%)'] = [
    f"{int(vordlus_kokku.loc[0, 'Tuhistamiste_osa_%'])}%",
    f"{int(vordlus_kokku.loc[1, 'Tuhistamiste_osa_%'])}%",
    f"{int(vordlus_kokku.loc[2, 'Tuhistamiste_osa_%'])}%" if vordlus_kokku.loc[2, 'Tuhistamiste_osa_%'] <= 0 else f"+{int(vordlus_kokku.loc[2, 'Tuhistamiste_osa_%'])}%"
]

kasv_str_2026 = f"+{int(kasv_protsent)}%" if kasv_protsent > 0 else f"{int(kasv_protsent)}%"
vaatamise_vordlus['Täidetud tellimuste kasv (%)'] = ["-", kasv_str_2026, "-"]

print("\n=== 2. VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026 ===")
display(vaatamise_vordlus)


=== 1. TÄIDETUD JA TÜHISTATUD TELLIMUSTE DÜNAAMIKA ===


,Aasta,Täidetud tellimused,Tühistatud tellimused,Kokku tellimusi,Tühistamiste osakaal (%),Täidetud tellimuste kasv (%)
0,2020,1,8,9,89%,0%
1,2021,173,72,245,29%,+17200%
2,2022,516,128,644,20%,+198%
3,2023,582,177,759,23%,+13%
4,2024,591,229,820,28%,+2%
5,2025,906,282,1 188,24%,+53%
6,2026,1 253,181,1 434,13%,+38%
7,Kokku,4 022,1 077,5 099,21%,-



=== 2. VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026 ===


,Periood / Aasta,Täidetud tellimused,Tühistatud tellimused,Kokku tellimusi,Tühistamiste osakaal (%),Täidetud tellimuste kasv (%)
0,01.01 - 17.09.2025,454,135,589,23%,-
1,01.01 - 17.09.2026,1 253,181,1 434,13%,+176%
2,Muutus (2026 vs 2025),+799,+46,+845,-10%,-


### 1.4 Keskmise tellimuse väärtuse dünaamika ###

In [7]:

# ==========================================
# 0. ANDMETE ETTEVALMISTUS JA FILTREERIMINE
# ==========================================
# Ühendame faktitabeli kuupäevade dimensiooniga, et saada ligipääs aastatele, kuudele ja päevadele
df = fact_order_items_clean.merge(
    dim_date[['date_id', 'year', 'm', 'day']], on='date_id', how='left'
)

# Filtreerime välja ainult edukalt lõpetatud tellimused (status == 'complete')
df_complete = df[df['status'].astype(str).str.strip().str.lower() == 'complete'].copy()


# ==========================================
# OSA 1: KESKMISE TELLIMUSE VÄÄRTUSE (AOV) DÜNAAMIKA TÄISAASTATE LÕIKES
# ==========================================
# Agregeerime andmed unikaalse tellimuse tasandile (kuna algandmed on toote tasemel)
order_totals = df_complete.groupby(['year', 'order_id'])['revenue_without_vat'].sum().reset_index()

# Arvutame aastate lõikes kogukäibe, tellimuste arvu ja keskmise tellimuse väärtuse (AOV)
aasta_koond = order_totals.groupby('year').agg(
    Käive=('revenue_without_vat', 'sum'),
    Tellimusi=('order_id', 'count'),       # Unikaalsete tellimuste arv grupis
    AOV=('revenue_without_vat', 'mean')    # Keskmine tellimuse väärtus
).reset_index()

# Ümardame AOV kahe komakohani ja arvutame kasvumäära (%)
aasta_koond['AOV'] = aasta_koond['AOV'].round(2)
aasta_koond['Kasv_%'] = aasta_koond['AOV'].pct_change() * 100
aasta_koond['Kasv_%'] = aasta_koond['Kasv_%'].fillna(0).round(1)

# Arvutame kogu perioodi koondnäitajad (kogurida "Kokku")
all_orders_total = order_totals['revenue_without_vat'].sum()
all_orders_count = order_totals['order_id'].nunique()
keskmine_kokku = round(all_orders_total / all_orders_count, 2) if all_orders_count > 0 else 0

kokku_rida = pd.DataFrame({
    'year': ['Kokku'],
    'Käive': [all_orders_total],
    'Tellimusi': [all_orders_count],
    'AOV': [keskmine_kokku],
    'Kasv_%': [None]
})
tulemus_aastad = pd.concat([aasta_koond, kokku_rida], ignore_index=True)

# Vormindame tabeli esitluse jaoks
vaata_aastad = pd.DataFrame()
vaata_aastad['Aasta'] = tulemus_aastad['year'].astype(str)
vaata_aastad['Keskmine tellimuse väärtus (EUR)'] = tulemus_aastad['AOV'].apply(lambda x: f"{x:,.2f} €".replace(",", " ") if pd.notnull(x) else "")
vaata_aastad['Kasvumäär (%)'] = tulemus_aastad['Kasv_%'].apply(
    lambda x: f"+{x}%" if pd.notnull(x) and x > 0 else (f"{x}%" if pd.notnull(x) and x < 0 else ("0%" if pd.notnull(x) else "-"))
)

print("=== 1. KESKMISE TELLIMUSE VÄÄRTUSE DÜNAAMIKA TÄISAASTATE LÕIKES ===")
display(vaata_aastad[['Aasta', 'Keskmine tellimuse väärtus (EUR)', 'Kasvumäär (%)']])


# ==========================================
# OSA 2: VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026
# ==========================================
# Määrame perioodi filtrid (vahemik 01. jaanuar kuni 17. september)
perioodi_filter = (
    ((df_complete['year'] == 2025) | (df_complete['year'] == 2026)) & 
    ((df_complete['m'] < 9) | ((df_complete['m'] == 9) & (df_complete['day'] <= 17)))
)
df_periood = df_complete[perioodi_filter].copy()

# Agregeerime samuti tellimuse tasandil antud perioodi jaoks
order_totals_periood = df_periood.groupby(['year', 'order_id'])['revenue_without_vat'].sum().reset_index()

periood_koond = order_totals_periood.groupby('year').agg(
    Käive=('revenue_without_vat', 'sum'),
    Tellimusi=('order_id', 'count'),
    AOV=('revenue_without_vat', 'mean')
).reset_index()

periood_koond['AOV'] = periood_koond['AOV'].round(2)

# Võtame eraldi väärtused 2025. ja 2026. aasta kohta
aov_2025 = periood_koond.loc[periood_koond['year'] == 2025, 'AOV'].values[0] if 2025 in periood_koond['year'].values else 0
aov_2026 = periood_koond.loc[periood_koond['year'] == 2026, 'AOV'].values[0] if 2026 in periood_koond['year'].values else 0

# Arvutame muutused eurodes ja protsentides
muutus_eur = round(aov_2026 - aov_2025, 2)
kasv_protsent = round((muutus_eur / aov_2025) * 100, 1) if aov_2025 > 0 else 0

# Koostame võrdlustabeli esitluse jaoks
vaata_vordlus = pd.DataFrame()
vaata_vordlus['Periood / Aasta'] = ['01.01 - 17.09.2025', '01.01 - 17.09.2026', 'Muutus (2026 vs 2025)']

aov_2025_str = f"{aov_2025:,.2f} €".replace(",", " ")
aov_2026_str = f"{aov_2026:,.2f} €".replace(",", " ")
muutus_str = f"+{muutus_eur:,.2f} €".replace(",", " ") if muutus_eur > 0 else f"{muutus_eur:,.2f} €".replace(",", " ")

vaata_vordlus['Keskmine tellimuse väärtus (EUR)'] = [aov_2025_str, aov_2026_str, muutus_str]

kasv_str_2026 = f"+{kasv_protsent}%" if kasv_protsent > 0 else f"{kasv_protsent}%"
vaata_vordlus['Kasvumäär (%)'] = ["-", kasv_str_2026, "-"]

print("\n=== 2. VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026 ===")
display(vaata_vordlus)

=== 1. KESKMISE TELLIMUSE VÄÄRTUSE DÜNAAMIKA TÄISAASTATE LÕIKES ===


,Aasta,Keskmine tellimuse väärtus (EUR),Kasvumäär (%)
0,2020,31.62 €,0%
1,2021,16.55 €,-47.7%
2,2022,24.33 €,+47.0%
3,2023,21.01 €,-13.6%
4,2024,20.35 €,-3.1%
5,2025,29.34 €,+44.2%
6,2026,38.67 €,+31.8%
7,Kokku,28.53 €,-



=== 2. VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026 ===


,Periood / Aasta,Keskmine tellimuse väärtus (EUR),Kasvumäär (%)
0,01.01 - 17.09.2025,23.99 €,-
1,01.01 - 17.09.2026,38.67 €,+61.2%
2,Muutus (2026 vs 2025),+14.68 €,-


### 1.5 Müüdud toodete arvu dünaamika ###

In [8]:
# 0. Andmete ettevalmistus: ühendame faktitabeli kuupäevade tabeliga
df = fact_order_items_clean.merge(
    dim_date[['date_id', 'year', 'm', 'day']], on='date_id', how='left'
)

# Filtreerime välja edukad tellimused (status == 'complete')
df['status_clean'] = df['status'].astype(str).str.strip().str.lower()
df_complete = df[df['status_clean'] == 'complete'].copy()

# Määrame kindla aastate vahemiku, et ükski aasta ei kaduks ära (saate vajadusel aastaid muuta)
kõik_aastad = sorted(df_complete['year'].dropna().unique())

# ==========================================
# OSA 1: Müüdud toodete arvu dünaamika (täisaastad)
# ==========================================
# Kasutame rexit 'quantity' (või 'qty' / 'amount' - vaadake, mis veeru nimi teil koguse jaoks on)
# Oletame, et koguse veeru nimi on 'quantity' või 'qty'. Kui teil on teine nimi, asendage see.
koguse_veeru_nimi = 'quantity' if 'quantity' in df_complete.columns else ('qty' if 'qty' in df_complete.columns else df_complete.columns[0])

# Grupeerime aastate järgi ja summeerime kogused, tagades kõikide aastate olemasolu (reindex)
tooted_aastate_kaupa = df_complete.groupby('year')[koguse_veeru_nimi].sum().reindex(kõik_aastad, fill_value=0)

tulemus_df = pd.DataFrame()
tulemus_df['year'] = tooted_aastate_kaupa.index
tulemus_df['Müüdud tooteid kokku (tk)'] = tooted_aastate_kaupa.values

# Aastane kasvumäär (%)
tulemus_df['Kasv_%'] = tulemus_df['Müüdud tooteid kokku (tk)'].pct_change() * 100
tulemus_df['Kasv_%'] = tulemus_df['Kasv_%'].fillna(0).round(0)

tulemus_df = tulemus_df.sort_values('year').reset_index(drop=True)

# Kokku rida
kokku_tooteid = tulemus_df['Müüdud tooteid kokku (tk)'].sum()

kokku_rida = pd.DataFrame({
    'year': ['Kokku'],
    'Müüdud tooteid kokku (tk)': [kokku_tooteid],
    'Kasv_%': [None]
})
tulemus_kokku = pd.concat([tulemus_df, kokku_rida], ignore_index=True)

vaatamise_tabel = pd.DataFrame()
vaatamise_tabel['Aasta'] = tulemus_kokku['year'].astype(str)
vaatamise_tabel['Müüdud tooteid kokku (tk)'] = tulemus_kokku['Müüdud tooteid kokku (tk)'].apply(lambda x: f"{int(x):,}".replace(",", " ") if pd.notnull(x) else "")
vaatamise_tabel['Kasvumäär (%)'] = tulemus_kokku['Kasv_%'].apply(
    lambda x: f"+{int(x)}%" if pd.notnull(x) and x > 0 else (f"{int(x)}%" if pd.notnull(x) and x < 0 else ("0%" if pd.notnull(x) else "-"))
)

print("=== 1.5 MÜÜDUD TOODETE ARVU DÜNAAMIKA (STATUS: COMPLETE) ===")
display(vaatamise_tabel)


# ==========================================
# OSA 2: Võrdlus sama perioodiga (01.01 - 17.09) 2025 vs 2026
# ==========================================
perioodi_filter = (
    ((df_complete['year'] == 2025) | (df_complete['year'] == 2026)) & 
    ((df_complete['m'] < 9) | ((df_complete['m'] == 9) & (df_complete['day'] <= 17)))
)
df_periood = df_complete[perioodi_filter].copy()

periood_kokkuvote = df_periood.groupby('year')[koguse_veeru_nimi].sum().reindex([2025, 2026], fill_value=0)

vordlus_df = pd.DataFrame()
vordlus_df['year'] = [2025, 2026]
vordlus_df['Müüdud tooteid kokku (tk)'] = periood_kokkuvote.values

tooteid_2025 = vordlus_df.loc[vordlus_df['year'] == 2025, 'Müüdud tooteid kokku (tk)'].values[0]
tooteid_2026 = vordlus_df.loc[vordlus_df['year'] == 2026, 'Müüdud tooteid kokku (tk)'].values[0]
muutus_tooteid = int(tooteid_2026 - tooteid_2025)
kasv_protsent_periood = int(round((muutus_tooteid / tooteid_2025) * 100, 0)) if tooteid_2025 > 0 else 0

muutuse_rida = pd.DataFrame({
    'year': ['Muutus'],
    'Müüdud tooteid kokku (tk) -> Muutus': [muutus_tooteid] # для отладки структуры
})

# Vormindame võrdlustabeli
vaatamise_vordlus = pd.DataFrame()
vaatamise_vordlus['Periood / Aasta'] = ['01.01 - 17.09.2025', '01.01 - 17.09.2026', 'Muutus (2026 vs 2025)']

def format_val(val, is_muutus=False):
    if pd.isnull(val):
        return ""
    v = int(val)
    if is_muutus and v > 0:
        return f"+{v:,}".replace(",", " ")
    return f"{v:,}".replace(",", " ")

vaatamise_vordlus['Müüdud tooteid kokku (tk)'] = [
    format_val(tooteid_2025),
    format_val(tooteid_2026),
    format_val(muutus_tooteid, is_muutus=True)
]

kasv_str_2026 = f"+{kasv_protsent_periood}%" if kasv_protsent_periood > 0 else f"{kasv_protsent_periood}%"
vaatamise_vordlus['Kasvumäär (%)'] = ["-", kasv_str_2026, "-"]

print("\n=== VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026 ===")
display(vaatamise_vordlus)

=== 1.5 MÜÜDUD TOODETE ARVU DÜNAAMIKA (STATUS: COMPLETE) ===


,Aasta,Müüdud tooteid kokku (tk),Kasvumäär (%)
0,2020,1,0%
1,2021,667,+66600%
2,2022,3 735,+460%
3,2023,2 337,-37%
4,2024,2 063,-12%
5,2025,3 738,+81%
6,2026,6 710,+80%
7,Kokku,19 251,-



=== VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026 ===


,Periood / Aasta,Müüdud tooteid kokku (tk),Kasvumäär (%)
0,01.01 - 17.09.2025,1 837,-
1,01.01 - 17.09.2026,6 710,+265%
2,Muutus (2026 vs 2025),+4 873,-


### 1.6 Aktiivsete ostjate arvu dünaamika ja osakaalud (Registreeritud vs Registreerimata kliendid) ###

In [9]:
# 0. Andmete ettevalmistus: ühendame faktitabeli kuupäevade tabeliga
df = fact_order_items_clean.merge(
    dim_date[['date_id', 'year', 'm', 'day']], on='date_id', how='left'
)

# Filtreerime välja edukad tellimused (status == 'complete')
df_complete = df[df['status'].astype(str).str.strip().str.lower() == 'complete'].copy()

# Puhastame customer_type veergu (väiketähed, tühikud)
df_complete['customer_type_clean'] = df_complete['customer_type'].astype(str).str.strip().str.lower()


# ==========================================
# OSA 1: Ostnud klientide arvu dünaamika ja osakaalud (täisaastad)
# ==========================================
klientide_kokkuvote = df_complete.groupby(['year', 'customer_type_clean'])['order_id'].nunique().unstack(fill_value=0)

tulemus_df = pd.DataFrame()
tulemus_df['year'] = klientide_kokkuvote.index

reg_col = [c for c in klientide_kokkuvote.columns if 'reg' in c and 'un' not in c and 'non' not in c]
unreg_col = [c for c in klientide_kokkuvote.columns if 'un' in c or 'guest' in c or 'non' in c]

tulemus_df['Registreeritud'] = klientide_kokkuvote[reg_col[0]].values if reg_col else 0
tulemus_df['Registreerimata'] = klientide_kokkuvote[unreg_col[0]].values if unreg_col else 0
tulemus_df['Kokku'] = tulemus_df['Registreeritud'] + tulemus_df['Registreerimata']

# Arvutame osakaalud protsentides
tulemus_df['Reg_%'] = (tulemus_df['Registreeritud'] / tulemus_df['Kokku'] * 100).round(0)
tulemus_df['Unreg_%'] = (tulemus_df['Registreerimata'] / tulemus_df['Kokku'] * 100).round(0)

# Aastane kasvumäär kokku klientidele
tulemus_df['Kasv_%'] = tulemus_df['Kokku'].pct_change() * 100
tulemus_df['Kasv_%'] = tulemus_df['Kasv_%'].fillna(0).round(0)

tulemus_df = tulemus_df.sort_values('year').reset_index(drop=True)

# Kokku rida
kokku_reg = tulemus_df['Registreeritud'].sum()
kokku_unreg = tulemus_df['Registreerimata'].sum()
kokku_kokku = tulemus_df['Kokku'].sum()
kokku_reg_osa = round((kokku_reg / kokku_kokku) * 100, 0) if kokku_kokku > 0 else 0
kokku_unreg_osa = round((kokku_unreg / kokku_kokku) * 100, 0) if kokku_kokku > 0 else 0

kokku_rida = pd.DataFrame({
    'year': ['Kokku'],
    'Registreeritud': [kokku_reg],
    'Registreerimata': [kokku_unreg],
    'Kokku': [kokku_kokku],
    'Reg_%': [kokku_reg_osa],
    'Unreg_%': [kokku_unreg_osa],
    'Kasv_%': [None]
})
tulemus_kokku = pd.concat([tulemus_df, kokku_rida], ignore_index=True)

vaatamise_tabel = pd.DataFrame()
vaatamise_tabel['Aasta'] = tulemus_kokku['year'].astype(str)
vaatamise_tabel['Registreeritud kliendid'] = tulemus_kokku['Registreeritud'].apply(lambda x: f"{int(x):,}".replace(",", " ") if pd.notnull(x) else "")
vaatamise_tabel['Registreeritud osakaal (%)'] = tulemus_kokku['Reg_%'].apply(lambda x: f"{int(x)}%" if pd.notnull(x) else "-")
vaatamise_tabel['Registreerimata kliendid'] = tulemus_kokku['Registreerimata'].apply(lambda x: f"{int(x):,}".replace(",", " ") if pd.notnull(x) else "")
vaatamise_tabel['Registreerimata osakaal (%)'] = tulemus_kokku['Unreg_%'].apply(lambda x: f"{int(x)}%" if pd.notnull(x) else "-")
vaatamise_tabel['Kliente kokku'] = tulemus_kokku['Kokku'].apply(lambda x: f"{int(x):,}".replace(",", " ") if pd.notnull(x) else "")
vaatamise_tabel['Kasvumäär (%)'] = tulemus_kokku['Kasv_%'].apply(
    lambda x: f"+{int(x)}%" if pd.notnull(x) and x > 0 else (f"{int(x)}%" if pd.notnull(x) and x < 0 else ("0%" if pd.notnull(x) else "-"))
)

print("=== 1. OSTNUD KLIENTIDE ARV JA OSAKAALUD TÄISAASTATE LÕIKES ===")
display(vaatamise_tabel)


# ==========================================
# OSA 2: Võrdlus sama perioodiga (01.01 - 17.09) 2025 vs 2026
# ==========================================
perioodi_filter = (
    ((df_complete['year'] == 2025) | (df_complete['year'] == 2026)) & 
    ((df_complete['m'] < 9) | ((df_complete['m'] == 9) & (df_complete['day'] <= 17)))
)
df_periood = df_complete[perioodi_filter].copy()

periood_kokkuvote = df_periood.groupby(['year', 'customer_type_clean'])['order_id'].nunique().unstack(fill_value=0)

vordlus_df = pd.DataFrame()
vordlus_df['year'] = [2025, 2026]

reg_2025 = periood_kokkuvote.loc[2025, reg_col[0]] if 2025 in periood_kokkuvote.index and reg_col and reg_col[0] in periood_kokkuvote.columns else 0
reg_2026 = periood_kokkuvote.loc[2026, reg_col[0]] if 2026 in periood_kokkuvote.index and reg_col and reg_col[0] in periood_kokkuvote.columns else 0
unreg_2025 = periood_kokkuvote.loc[2025, unreg_col[0]] if 2025 in periood_kokkuvote.index and unreg_col and unreg_col[0] in periood_kokkuvote.columns else 0
unreg_2026 = periood_kokkuvote.loc[2026, unreg_col[0]] if 2026 in periood_kokkuvote.index and unreg_col and unreg_col[0] in periood_kokkuvote.columns else 0

vordlus_df['Registreeritud'] = [reg_2025, reg_2026]
vordlus_df['Registreerimata'] = [unreg_2025, unreg_2026]
vordlus_df['Kokku'] = vordlus_df['Registreeritud'] + vordlus_df['Registreerimata']

vordlus_df['Reg_%'] = (vordlus_df['Registreeritud'] / vordlus_df['Kokku'] * 100).round(0)
vordlus_df['Unreg_%'] = (vordlus_df['Registreerimata'] / vordlus_df['Kokku'] * 100).round(0)

kokku_2025 = vordlus_df.loc[vordlus_df['year'] == 2025, 'Kokku'].values[0]
kokku_2026 = vordlus_df.loc[vordlus_df['year'] == 2026, 'Kokku'].values[0]
muutus_kokku = int(kokku_2026 - kokku_2025)
kasv_protsent = int(round((muutus_kokku / kokku_2025) * 100, 0)) if kokku_2025 > 0 else 0

muutus_reg = int(reg_2026 - reg_2025)
muutus_unreg = int(unreg_2026 - unreg_2025)

muutus_reg_osa = vordlus_df.loc[1, 'Reg_%'] - vordlus_df.loc[0, 'Reg_%']
muutus_unreg_osa = vordlus_df.loc[1, 'Unreg_%'] - vordlus_df.loc[0, 'Unreg_%']

muutuse_rida = pd.DataFrame({
    'year': ['Muutus'],
    'Registreeritud': [muutus_reg],
    'Registreerimata': [muutus_unreg],
    'Kokku': [muutus_kokku],
    'Reg_%': [muutus_reg_osa],
    'Unreg_%': [muutus_unreg_osa]
})
vordlus_kokku = pd.concat([vordlus_df, muutuse_rida], ignore_index=True)

vaatamise_vordlus = pd.DataFrame()
vaatamise_vordlus['Periood / Aasta'] = ['01.01 - 17.09.2025', '01.01 - 17.09.2026', 'Muutus (2026 vs 2025)']

def format_val(val, is_muutus=False):
    if pd.isnull(val):
        return ""
    v = int(val)
    if is_muutus and v > 0:
        return f"+{v:,}".replace(",", " ")
    return f"{v:,}".replace(",", " ")

vaatamise_vordlus['Registreeritud kliendid'] = [
    format_val(vordlus_kokku.loc[0, 'Registreeritud']),
    format_val(vordlus_kokku.loc[1, 'Registreeritud']),
    format_val(vordlus_kokku.loc[2, 'Registreeritud'], is_muutus=True)
]
vaatamise_vordlus['Registreeritud osakaal (%)'] = [
    f"{int(vordlus_kokku.loc[0, 'Reg_%'])}%",
    f"{int(vordlus_kokku.loc[1, 'Reg_%'])}%",
    f"{int(muutus_reg_osa)}% pt" if muutus_reg_osa < 0 else f"+{int(muutus_reg_osa)}% pt"
]
vaatamise_vordlus['Registreerimata kliendid'] = [
    format_val(vordlus_kokku.loc[0, 'Registreerimata']),
    format_val(vordlus_kokku.loc[1, 'Registreerimata']),
    format_val(vordlus_kokku.loc[2, 'Registreerimata'], is_muutus=True)
]
vaatamise_vordlus['Registreerimata osakaal (%)'] = [
    f"{int(vordlus_kokku.loc[0, 'Unreg_%'])}%",
    f"{int(vordlus_kokku.loc[1, 'Unreg_%'])}%",
    f"{int(muutus_unreg_osa)}% pt" if muutus_unreg_osa < 0 else f"+{int(muutus_unreg_osa)}% pt"
]
vaatamise_vordlus['Kliente kokku'] = [
    format_val(vordlus_kokku.loc[0, 'Kokku']),
    format_val(vordlus_kokku.loc[1, 'Kokku']),
    format_val(vordlus_kokku.loc[2, 'Kokku'], is_muutus=True)
]

kasv_str_2026 = f"+{kasv_protsent}%" if kasv_protsent > 0 else f"{kasv_protsent}%"
vaatamise_vordlus['Kasvumäär (%)'] = ["-", kasv_str_2026, "-"]

print("\n=== 2. VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026 ===")
display(vaatamise_vordlus)

=== 1. OSTNUD KLIENTIDE ARV JA OSAKAALUD TÄISAASTATE LÕIKES ===


,Aasta,Registreeritud kliendid,Registreeritud osakaal (%),Registreerimata kliendid,Registreerimata osakaal (%),Kliente kokku,Kasvumäär (%)
0,2020,0,0%,1,100%,1,0%
1,2021,21,12%,152,88%,173,+17200%
2,2022,92,18%,424,82%,516,+198%
3,2023,104,18%,478,82%,582,+13%
4,2024,165,28%,426,72%,591,+2%
5,2025,293,32%,613,68%,906,+53%
6,2026,476,38%,777,62%,1 253,+38%
7,Kokku,1 151,29%,2 871,71%,4 022,-



=== 2. VÕRDLUS SAMA PERIOODIGA (01.01 - 17.09) 2025 vs 2026 ===


,Periood / Aasta,Registreeritud kliendid,Registreeritud osakaal (%),Registreerimata kliendid,Registreerimata osakaal (%),Kliente kokku,Kasvumäär (%)
0,01.01 - 17.09.2025,130,29%,324,71%,454,-
1,01.01 - 17.09.2026,476,38%,777,62%,1 253,+176%
2,Muutus (2026 vs 2025),+346,+9% pt,+453,-9% pt,+799,-


## 2.Ostukäitumine ##

### 2.1: Ühekordse ja korduvostuga klientide ostukäitumine ###

In [10]:
# 0. Filtreerime välja ainult täidetud tellimused (et numbrid klapiksid Power BI-ga)
orders = fact_order_items_clean[fact_order_items_clean['status'] == 'complete'].copy()

# 1. Arvutame välja unikaalsete tellimuste arvu iga kliendi kohta edukate tellimuste põhjal
klientide_tellimuste_arv = orders.groupby('customer_id')['order_id'].nunique().reset_index()
klientide_tellimuste_arv.columns = ['customer_id', 'tegelik_tellimuste_arv']

# 2. Klassifitseerime kliendid tegelike edukate tellimuste alusel (suure algustähega)
klientide_tellimuste_arv['Klientide_segment'] = klientide_tellimuste_arv['tegelik_tellimuste_arv'].apply(
    lambda x: 'Ühekordne klient' if x == 1 else 'Korduvklient'
)

# 3. Ühendame segmendid tellimustega
tellimused_segmendiga = orders.merge(
    klientide_tellimuste_arv[['customer_id', 'Klientide_segment']], 
    on='customer_id', 
    how='left'
)

# 4. Arvutame tellimuste summad ilma käibemaksuta
tellimuse_summad = tellimused_segmendiga.groupby(['order_id', 'customer_id', 'Klientide_segment']).agg({
    'revenue_without_vat': 'sum'
}).reset_index()

# 5. Agregeerime andmed põhinäitajate saamiseks
segmendi_koond = tellimuse_summad.groupby('Klientide_segment').agg(
    Klientide_arv=('customer_id', 'nunique'),
    Tellimuste_arv=('order_id', 'count'),
    Käive_ilma_km_EUR=('revenue_without_vat', 'sum')
).reset_index()

# --- LISAME UUED ANALÜÜTILISED NÄITAJAD ---

# Kokkuvõtvad väärtused osakaalude arvutamiseks
kogu_kliente = klientide_tellimuste_arv['customer_id'].nunique()
kogukaive = segmendi_koond['Käive_ilma_km_EUR'].sum()
kogutellimusi = segmendi_koond['Tellimuste_arv'].sum()

# 1. Kliendibaasi osakaal protsentides (%) koos % märgiga (ümardatud täisarvuks)
segmendi_koond['Klientide_osakaal_%'] = (
    ((segmendi_koond['Klientide_arv'] / kogu_kliente) * 100).round(0).astype(int).astype(str) + '%'
)

# 2. Keskmine tellimuste arv ühe kliendi kohta segmendis
segmendi_koond['Keskmine_tellimuste_arv'] = segmendi_koond['Tellimuste_arv'] / segmendi_koond['Klientide_arv']

# 3. Keskmine käive ühe kliendi kohta (Revenue per customer / LTV element)
segmendi_koond['Käive_klient_kohta_EUR'] = segmendi_koond['Käive_ilma_km_EUR'] / segmendi_koond['Klientide_arv']

# 4. Keskmine tellimus (keskmine tšekk)
segmendi_koond['Keskmine_tellimus_ilma_km_EUR'] = segmendi_koond['Käive_ilma_km_EUR'] / segmendi_koond['Tellimuste_arv']

# 5. Käibe osakaal protsentides (%) koos % märgiga (ümardatud täisarvuks)
segmendi_koond['Käibe_osakaal_%'] = (
    ((segmendi_koond['Käive_ilma_km_EUR'] / kogukaive) * 100).round(0).astype(int).astype(str) + '%'
)

# Järjestame veerud loogilisse ja esinduslikku järjekorda
segmendi_koond = segmendi_koond[[
    'Klientide_segment', 
    'Klientide_arv', 
    'Klientide_osakaal_%', 
    'Tellimuste_arv', 
    'Keskmine_tellimuste_arv', 
    'Käive_ilma_km_EUR', 
    'Käive_klient_kohta_EUR', 
    'Keskmine_tellimus_ilma_km_EUR', 
    'Käibe_osakaal_%'
]]

# Ümardame KÕIK numbrilised veerud täisarvudeks (0 kohta pärast koma)
segmendi_koond['Klientide_arv'] = segmendi_koond['Klientide_arv'].round(0).astype(int)
segmendi_koond['Tellimuste_arv'] = segmendi_koond['Tellimuste_arv'].round(0).astype(int)
segmendi_koond['Keskmine_tellimuste_arv'] = segmendi_koond['Keskmine_tellimuste_arv'].round(0).astype(int)
segmendi_koond['Käive_ilma_km_EUR'] = segmendi_koond['Käive_ilma_km_EUR'].round(0).astype(int)
segmendi_koond['Käive_klient_kohta_EUR'] = segmendi_koond['Käive_klient_kohta_EUR'].round(0).astype(int)
segmendi_koond['Keskmine_tellimus_ilma_km_EUR'] = segmendi_koond['Keskmine_tellimus_ilma_km_EUR'].round(0).astype(int)

# --- KOKKUVÕTTE RIDA (TOTAL) ---
keskmine_tellimus_kokku = round(kogukaive / kogutellimusi, 0) if kogutellimusi > 0 else 0
keskmine_tell_arv_kokku = round(kogutellimusi / kogu_kliente, 0) if kogu_kliente > 0 else 0
kaive_klient_kohta_kokku = round(kogukaive / kogu_kliente, 0) if kogu_kliente > 0 else 0

kokku_rida = pd.DataFrame({
    'Klientide_segment': ['KOKKU'],
    'Klientide_arv': [int(kogu_kliente)],
    'Klientide_osakaal_%': ['100%'],
    'Tellimuste_arv': [int(kogutellimusi)],
    'Keskmine_tellimuste_arv': [int(keskmine_tell_arv_kokku)],
    'Käive_ilma_km_EUR': [int(round(kogukaive, 0))],
    'Käive_klient_kohta_EUR': [int(kaive_klient_kohta_kokku)],
    'Keskmine_tellimus_ilma_km_EUR': [int(keskmine_tellimus_kokku)],
    'Käibe_osakaal_%': ['100%']
})

# Nimetame veeru ümber, et klapiks
kokku_rida.columns = segmendi_koond.columns

# Liidame kokkuvõttereas
segmendi_koond_tulemus = pd.concat([segmendi_koond, kokku_rida], ignore_index=True)

# Kuvame tulemuse ekraanile
display(segmendi_koond_tulemus)

,Klientide_segment,Klientide_arv,Klientide_osakaal_%,Tellimuste_arv,Keskmine_tellimuste_arv,Käive_ilma_km_EUR,Käive_klient_kohta_EUR,Keskmine_tellimus_ilma_km_EUR,Käibe_osakaal_%
0,Korduvklient,498,15%,1301,3,39446,79,30,34%
1,Ühekordne klient,2721,85%,2721,1,75289,28,28,66%
2,KOKKU,3219,100%,4022,1,114734,36,29,100%


### 2.2	Ostusagedus ja ostude korduvus ###

In [11]:
# 0. Filtreerime välja ainult edukad tellimused (status == 'complete')
orders = fact_order_items_clean[fact_order_items_clean['status'] == 'complete'].copy()

# 1. Leiame iga korduvkliendi kohta tellimuste arvud ja kuupäevad
klientide_stats = orders.groupby('customer_id').agg(
    Ostude_arv=('order_id', 'nunique'),
    Esimene_kuupäev=('date_id', 'min'),
    Viimane_kuupäev=('date_id', 'max')
).reset_index()

# 2. Jätame alles AINULT need kliendid, kes on korduvkliendid (ostude arv > 1)
korduv_kliendid = klientide_stats[klientide_stats['Ostude_arv'] > 1].copy()

# 3. Leiame iga kliendi jaoks keskmise päevade arvu ostude vahel
if 'days_between_last_order' in fact_order_items_clean.columns:
    keskmised_vahed = orders.groupby('customer_id')['days_between_last_order'].mean().reset_index()
    korduv_kliendid = korduv_kliendid.merge(keskmised_vahed, on='customer_id', how='left')
else:
    korduv_kliendid['days_between_last_order'] = 0

# 4. Jagame korduvkliendid alagruppidesse ostude sageduse järgi
def maaritle_sagedus(ostude_arv):
    if ostude_arv == 2:
        return '2 ostu'
    elif ostude_arv == 3:
        return '3 ostu'
    else:
        return '4+ ostu'

korduv_kliendid['Sagedus_grupp'] = korduv_kliendid['Ostude_arv'].apply(maaritle_sagedus)

# 5. Agregeerime andmed sagedusgruppide kaupa
analuus = korduv_kliendid.groupby('Sagedus_grupp').agg(
    Klientide_arv=('customer_id', 'count'),
    Tellimuste_arv=('Ostude_arv', 'sum'),
    Keskmine_vahe_paevades=('days_between_last_order', 'mean')
).reset_index()

# Kokkuvõttev korduvklientide arv
kogu_korduv_kliente = korduv_kliendid['customer_id'].nunique()

# Arvutame osakaalud % korduvklientide koguarvust ja lisame % märgistuse (ümardatud täisarvuks)
analuus['Osakaal_korduvklientidest_%'] = (
    ((analuus['Klientide_arv'] / kogu_korduv_kliente) * 100).round(0).astype(int).astype(str) + '%'
)

# Keskmine tellimuste arv ühe kliendi kohta grupis
analuus['Keskmine_tellimuste_arv_kliendi_kohta'] = analuus['Tellimuste_arv'] / analuus['Klientide_arv']

# Järjestame veerud loogiliselt (Sagedus_grupp suure tähega)
analuus = analuus[[
    'Sagedus_grupp', 
    'Klientide_arv', 
    'Osakaal_korduvklientidest_%', 
    'Tellimuste_arv', 
    'Keskmine_tellimuste_arv_kliendi_kohta', 
    'Keskmine_vahe_paevades'
]]

# Ümardame ülejäänud numbrilised veerud täisarvudeks (0 kohta pärast koma)
analuus['Klientide_arv'] = analuus['Klientide_arv'].round(0).astype(int)
analuus['Tellimuste_arv'] = analuus['Tellimuste_arv'].round(0).astype(int)
analuus['Keskmine_tellimuste_arv_kliendi_kohta'] = analuus['Keskmine_tellimuste_arv_kliendi_kohta'].round(0).astype(int)
analuus['Keskmine_vahe_paevades'] = analuus['Keskmine_vahe_paevades'].round(0).astype(int)

# --- LISANDUS: Kokkuvõtte rida (Total) ---
kokku_kliente = korduv_kliendid['customer_id'].nunique()
kokku_tellimusi = korduv_kliendid['Ostude_arv'].sum()
keskmine_vahe_kokku = round(korduv_kliendid['days_between_last_order'].mean(), 0)
keskmine_tell_arv_kokku = round(kokku_tellimusi / kokku_kliente, 0) if kokku_kliente > 0 else 0

kokku_rida = pd.DataFrame({
    'Sagedus_grupp': ['Kokku korduvkliente'],
    'Klientide_arv': [int(kokku_kliente)],
    'Osakaal_korduvklientidest_%': ['100%'],
    'Tellimuste_arv': [int(kokku_tellimusi)],
    'Keskmine_tellimuste_arv_kliendi_kohta': [int(keskmine_tell_arv_kokku)],
    'Keskmine_vahe_paevades': [int(keskmine_vahe_kokku)]
})

# Liidame kokkuvõttereas
korduv_analuus_tulemus = pd.concat([analuus, kokku_rida], ignore_index=True)

print(f"=== KORDUVKLIENTIDE OSTUSAGEDUSE JA KORDUVUSE ANALÜÜS ===")
display(korduv_analuus_tulemus)

=== KORDUVKLIENTIDE OSTUSAGEDUSE JA KORDUVUSE ANALÜÜS ===


,Sagedus_grupp,Klientide_arv,Osakaal_korduvklientidest_%,Tellimuste_arv,Keskmine_tellimuste_arv_kliendi_kohta,Keskmine_vahe_paevades
0,2 ostu,345,69%,690,2,215
1,3 ostu,83,17%,249,3,218
2,4+ ostu,70,14%,362,5,149
3,Kokku korduvkliente,498,100%,1301,3,206


### 2.3 Ostukorvi analüüs ###

In [12]:
# 1. Ühendame ostukorvid kalendriga
df_cart_date = fact_cart_items.merge(dim_date, on='date_id', how='inner')

# Määrame uuritava perioodi alguse ja lõpu kuupäevad
min_date = df_cart_date['date'].min() if 'date' in df_cart_date.columns else f"{df_cart_date['year'].min()}-{df_cart_date['m'].min():02d}"
max_date = df_cart_date['date'].max() if 'date' in df_cart_date.columns else f"{df_cart_date['year'].max()}-{df_cart_date['m'].max():02d}"

print('--- ANALÜÜSI PERIOOD ---')
print(f'Uuritav periood (Analysis period): alates {min_date} kuni {max_date}\n')

# 2. Arvutame näitajad iga ostukorvi kohta eraldi (kogusumma ILMA käibemaksuta ja toodete arv)
cart_agg = (
    df_cart_date.groupby(['cart_id', 'status'])
    .agg(
        cart_value=('items_amount_without_vat', 'sum'),  # Ostukorvi maksumus ilma käibemaksuta
        cart_qty=('quantity', 'sum'),  # Toodete koguarv ostukorvis
    )
    .reset_index()
)

# 3. Grupeerime andmed staatuste kaupa (koondtabeli koostamine)
summary_table = (
    cart_agg.groupby('status')
    .agg(
        Ostukorvide_arv=('cart_id', 'count'),  # Ostukorvide arv igas staatuses
        Kogusumma_ilma_km_eur=('cart_value', 'sum'),  # Toodete kogusumma ilma käibemaksuta €
        Keskmine_väärtus_ilma_km_eur=('cart_value', 'mean'),  # Keskmine ostukorvi väärtus ilma km
        Keskmine_toodete_arv=('cart_qty', 'mean'),  # Keskmine toodete arv ostukorvis
    )
    .reset_index()
)

# Teeme staatuse nime esimese tähe suureks (Capitalize)
summary_table['status'] = summary_table['status'].str.capitalize()

# Arvutame osakaalu protsentides koguarvust ja ümardame täisarvuks
total_carts = summary_table['Ostukorvide_arv'].sum()
summary_table['Osakaal_protsentides'] = (
    (summary_table['Ostukorvide_arv'] / total_carts) * 100
).round(0).astype(int).astype(str) + '%'

# Seome veerud loogilisse järjekorda
summary_table = summary_table[
    [
        'status',
        'Ostukorvide_arv',
        'Osakaal_protsentides',
        'Kogusumma_ilma_km_eur',
        'Keskmine_väärtus_ilma_km_eur',
        'Keskmine_toodete_arv',
    ]
]

# 4. Loome kokkuvõtva rea (KOKKU / TOTAL)
total_row = pd.DataFrame(
    {
        'status': ['KOKKU / TOTAL'],
        'Ostukorvide_arv': [total_carts],
        'Osakaal_protsentides': ['100%'],
        'Kogusumma_ilma_km_eur': [cart_agg['cart_value'].sum()],
        'Keskmine_väärtus_ilma_km_eur': [cart_agg['cart_value'].mean()],
        'Keskmine_toodete_arv': [cart_agg['cart_qty'].mean()],
    }
)

final_report = pd.concat([summary_table, total_row], ignore_index=True)

# Ümardame ülejäänud numbrilised veerud täisarvudeks (0 kohta pärast koma)
final_report['Kogusumma_ilma_km_eur'] = final_report['Kogusumma_ilma_km_eur'].round(0).astype(int)
final_report['Keskmine_väärtus_ilma_km_eur'] = final_report['Keskmine_väärtus_ilma_km_eur'].round(0).astype(int)
final_report['Keskmine_toodete_arv'] = final_report['Keskmine_toodete_arv'].round(0).astype(int)

# Väljastame tabeli kujul, mida saab hõlpsasti Excelisse kopeerida
print('--- OSTUKORVI ANALÜÜSI KOONDTABEL (ILMA KÄIBEMAKSUTA) ---')
display(final_report)

--- ANALÜÜSI PERIOOD ---
Uuritav periood (Analysis period): alates 2026-07 kuni 2026-09

--- OSTUKORVI ANALÜÜSI KOONDTABEL (ILMA KÄIBEMAKSUTA) ---


,status,Ostukorvide_arv,Osakaal_protsentides,Kogusumma_ilma_km_eur,Keskmine_väärtus_ilma_km_eur,Keskmine_toodete_arv
0,Abandoned,812,74%,18678,23,2
1,Active,112,10%,2060,18,3
2,Converted,179,16%,6227,35,5
3,KOKKU / TOTAL,1103,100%,26966,24,3


## 3.	Kliendibaasi analüüs ##

### 3.1 Kliendibaasi kasv aastate lõikes ###

In [16]:
# 0. Ühendame faktitabeli Dim_date tabeliga, et saada kätte õige aasta
orders = fact_order_items_clean.merge(dim_date[['date_id', 'year']], on='date_id', how='left')

# Filtreerime välja ainult edukad tellimused (status == 'complete')
# Kasutame turvalist puhastust juhuks, kui on tühikuid või suuri tähti
orders['status_clean'] = orders['status'].astype(str).str.strip().str.lower()
orders = orders[orders['status_clean'] == 'complete'].copy()

# Määrame aasta veeru
orders['Year'] = orders['year']

# 1. Leiame iga kliendi esimese tellimuse aasta (millal klient liitus)
klientide_esimene_ost = orders.groupby('customer_id')['Year'].min().reset_index()

# 2. Grupeerime aastate kaupa ja loendame uute klientide arvu
uued_kliendid_aastas = klientide_esimene_ost.groupby('Year').size().reset_index(name='Uute_klientide_arv')
uued_kliendid_aastas = uued_kliendid_aastas.sort_values('Year').reset_index(drop=True)

# 3. Arvutame aastase kasvu protsentides võrreldes eelneva aastaga (Kasv_%)
uued_kliendid_aastas['Kasv_%'] = uued_kliendid_aastas['Uute_klientide_arv'].pct_change() * 100
uued_kliendid_aastas['Kasv_%'] = uued_kliendid_aastas['Kasv_%'].fillna(0).round(0).astype(int)

# --- KONKREETSED KUUPÄEVAD TEKSTI JAOKS ---
esimene_kuupäev_tekst = "15.12.2020" 
viimane_kuupäev_tekst = "17.09.2026" 

print(f"=== 3.1 KLIENDIBAASI KASV AASTATE LÕIKES ===")
print(f"E-poe esimene tellimus tehti: {esimene_kuupäev_tekst} | Analüüsitud periood kuni: {viimane_kuupäev_tekst} (2026. aasta andmed on osalised)\n")

# --- Arvutame kokku uute klientide arvu ---
kokku_uusi_kliente = int(uued_kliendid_aastas['Uute_klientide_arv'].sum())

# Valmistame tulemustabeli (muudame tekstid ja veerud eesti keelde)
analuus_tulemus = pd.DataFrame()
analuus_tulemus['Aasta'] = uued_kliendid_aastas['Year'].astype(str)
analuus_tulemus['Uute klientide arv'] = uued_kliendid_aastas['Uute_klientide_arv'].apply(lambda x: f"{int(x):,}".replace(",", " "))

# Vormindame kasvumäära ilusa plussmärgiga
analuus_tulemus['Kasvumäär (%)'] = uued_kliendid_aastas['Kasv_%'].apply(
    lambda x: f"+{int(x)}%" if x > 0 else (f"{int(x)}%" if x < 0 else "0%")
)

# Loome kokkuvõtterea (Total row) eesti keeles
kokku_rida = pd.DataFrame({
    'Aasta': ['Kokku uusi kliente'],
    'Uute klientide arv': [f"{int(kokku_uusi_kliente):,}".replace(",", " ")],
    'Kasvumäär (%)': ['100%']
})

# Liidame põhitabeli ja kokkuvõttereas kokku
lõplik_tulemus = pd.concat([analuus_tulemus, kokku_rida], ignore_index=True)

# Kuvame lõpliku tabeli
display(lõplik_tulemus)

=== 3.1 KLIENDIBAASI KASV AASTATE LÕIKES ===
E-poe esimene tellimus tehti: 15.12.2020 | Analüüsitud periood kuni: 17.09.2026 (2026. aasta andmed on osalised)



,Aasta,Uute klientide arv,Kasvumäär (%)
0,2020,1,0%
1,2021,163,+16200%
2,2022,408,+150%
3,2023,442,+8%
4,2024,466,+5%
5,2025,750,+61%
6,2026,989,+32%
7,Kokku uusi kliente,3 219,100%


### RFM-analüüs ja kliendisegmentide moodustamine ###

In [17]:
import pandas as pd
from datetime import datetime

# 1. Filtreerime välja ainult edukad tellimused (status = complete)
df_complete = fact_order_items_clean[fact_order_items_clean['status'].str.lower() == 'complete'].copy()

# 2. Teisendame tellimuse kuupäeva formaati datetime (kujul YYYYMMDD)
df_complete['order_date'] = pd.to_datetime(df_complete['date_id'].astype(str), format='%Y%m%d')

# 3. Määrame "tänase" kuupäeva (snapshot_date) Recency arvutamiseks
snapshot_date = df_complete['order_date'].max() + pd.Timedelta(days=1)

# 4. Arvutame igale kliendile R, F ja M näitajad
rfm_df = df_complete.groupby('customer_id').agg(
    recency=('order_date', lambda x: (snapshot_date - x.max()).days),  # Värskus: Päevad viimasest ostust 
    frequency=('order_id', 'nunique'),                                    # Sagedus: Unikaalsete tellimuste arv 
    monetary=('revenue_without_vat', 'sum')                                # Käive: Kogu kulutatud summa (ilma käibemaksuta)
).reset_index()

# 5. Anname klientidele punktid (skoorid) skaalal 1 kuni 5
rfm_df['R_score'] = pd.qcut(rfm_df['recency'], q=5, labels=[5, 4, 3, 2, 1])

def assign_f_score(freq):
    if freq == 1:
        return 1
    elif freq == 2:
        return 2
    elif freq == 3:
        return 3
    elif freq in [4, 5]:
        return 4
    else:
        return 5

rfm_df['F_score'] = rfm_df['frequency'].apply(assign_f_score)
rfm_df['M_score'] = pd.qcut(rfm_df['monetary'], q=5, labels=[1, 2, 3, 4, 5])

# 6. Loome ajutised tekstiveerud segmentide määramiseks (et kood töötaks korrektselt)
r_str = rfm_df['R_score'].astype(str)
f_str = rfm_df['F_score'].astype(str)
m_str = rfm_df['M_score'].astype(str)

rfm_df['RFM_Score'] = r_str + f_str + m_str

# 7. Segmenteerime kliendid rühmadesse
def assign_rfm_segment(row_r, row_f, row_m):
    r = int(row_r)
    f = int(row_f)
    m = int(row_m)
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Parimad kliendid'
    elif f >= 3 and m >= 3:
        return 'Lojaalsed kliendid'
    elif r >= 4 and f == 1:
        return 'Uued kliendid'
    elif r >= 3 and f >= 2:
        return 'Potentsiaalsed püsikliendid'
    elif r <= 2 and f >= 2:
        return 'Ohustatud kliendid'
    else:
        return 'Uinunud / Kaotatud kliendid'

rfm_df['Segment'] = [assign_rfm_segment(r, f, m) for r, f, m in zip(r_str, f_str, m_str)]

# 8. Teeme koondtabeli segmentide kaupa
segment_summary = rfm_df.groupby('Segment').agg(
    klientide_arv=('customer_id', 'count'),
    kogukäive=('monetary', 'sum'),
    keskmine_käive=('monetary', 'mean')
).reset_index()

# Arvutame protsendid
total_customers = segment_summary['klientide_arv'].sum()
segment_summary['klientide_osakaal_%'] = (segment_summary['klientide_arv'] / total_customers) * 100

# Seame segmendid soovitud loogilisse järjekorda
segment_order = [
    'Parimad kliendid', 
    'Lojaalsed kliendid', 
    'Uued kliendid', 
    'Potentsiaalsed püsikliendid', 
    'Ohustatud kliendid', 
    'Uinunud / Kaotatud kliendid'
]

segment_summary['Segment'] = pd.Categorical(segment_summary['Segment'], categories=segment_order, ordered=True)
segment_summary = segment_summary.sort_values('Segment').reset_index(drop=True)

# 9. Loome "Kokku" rea enne andmete ümardamist tekstiks
total_row = pd.DataFrame({
    'Segment': ['Kokku'],
    'klientide_arv': [segment_summary['klientide_arv'].sum()],
    'kogukäive': [segment_summary['kogukäive'].sum()],
    'keskmine_käive': [rfm_df['monetary'].mean()],
    'klientide_osakaal_%': [segment_summary['klientide_osakaal_%'].sum()]
})

segment_summary_with_total = pd.concat([segment_summary, total_row], ignore_index=True)

# 10. Vormindame numbrid (rahasummad täisarvudeks, protsendid ühe komakohaga ja '%' märgiga)
segment_summary_with_total['kogukäive'] = segment_summary_with_total['kogukäive'].round(0).astype(int)
segment_summary_with_total['keskmine_käive'] = segment_summary_with_total['keskmine_käive'].round(0).astype(int)
segment_summary_with_total['klientide_osakaal_%'] = segment_summary_with_total['klientide_osakaal_%'].round(1).astype(str) + ' %'

# 11. Nimetame veerud ilusasse eesti keelde diplomitöö jaoks
segment_summary_with_total.columns = [
    'Segment', 
    'Klientide arv (tk)', 
    'Kogukäive (€)', 
    'Keskmine käive (€)', 
    'Klientide osakaal'
]

# 12. Kuvame tulemused ekraanile
print("--- Kliendiandmed (puhastatud põhitabel) ---")
display(rfm_df.head(5))

print("\n--- Peamine RFM koondtabel segmentide kaupa (koos Kokku reaga) ---")
display(segment_summary_with_total)

# 13. Salvestame Exceli faili arvutisse (2 eraldi lehte ühes failis)
output_filename = 'rfm_diplom_analysis.xlsx'
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    rfm_df.to_excel(writer, sheet_name='Kliendid_detsember_RFM', index=False)
    segment_summary_with_total.to_excel(writer, sheet_name='Segmendid_koond', index=False)

print(f"\nFail '{output_filename}' on edukalt salvestatud!")

--- Kliendiandmed (puhastatud põhitabel) ---


,customer_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_Score,Segment
0,C00001,685,2,19.13,2,2,3,223,Ohustatud kliendid
1,C00002,1498,2,47.66,1,2,4,124,Ohustatud kliendid
2,C00003,1433,1,15.07,1,1,2,112,Uinunud / Kaotatud kliendid
3,C00004,1413,1,15.88,1,1,2,112,Uinunud / Kaotatud kliendid
4,C00005,341,1,15.72,3,1,2,312,Uinunud / Kaotatud kliendid



--- Peamine RFM koondtabel segmentide kaupa (koos Kokku reaga) ---


,Segment,Klientide arv (tk),Kogukäive (€),Keskmine käive (€),Klientide osakaal
0,Parimad kliendid,23,4703,204,0.7 %
1,Lojaalsed kliendid,124,13574,109,3.9 %
2,Uued kliendid,1078,39095,36,33.5 %
3,Potentsiaalsed püsikliendid,219,15502,71,6.8 %
4,Ohustatud kliendid,132,5666,43,4.1 %
5,Uinunud / Kaotatud kliendid,1643,36194,22,51.0 %
6,Kokku,3219,114734,36,100.0 %



Fail 'rfm_diplom_analysis.xlsx' on edukalt salvestatud!


### RFM-kriteeriumide määratlemine ###

In [15]:

import pandas as pd

# 1. Recency piiride arvutamine ja klientide arvu leidmine skooride kaupa
r_bins, r_bins_edges = pd.qcut(rfm_df['recency'], q=5, retbins=True, labels=[5, 4, 3, 2, 1])
r_counts = rfm_df['R_score'].value_counts().sort_index(ascending=False)

# 2. Monetary piiride arvutamine ja klientide arvu leidmine skooride kaupa
m_bins, m_bins_edges = pd.qcut(rfm_df['monetary'], q=5, retbins=True, labels=[1, 2, 3, 4, 5])
m_counts = rfm_df['M_score'].value_counts().sort_index()

# 3. Frequency klientide arvu leidmine skooride kaupa
f_counts = rfm_df['F_score'].value_counts().sort_index()

# Abifunktsioon päevade teisendamiseks aastateks (ühe kümnendkoha täpsusega)
def days_to_years(days):
    return round(days / 365, 1)

# Kogumeandmed ühtsesse struktuuri, kus KÕIK mõõdikud on järjestatud 5-st 1-ni
criteria_list = [
    # --- RECENCY (5 kuni 1) ---
    {"Mõõdik": "Recency (Värskus)", "Skoor": 5, "Vahemik": f"{r_bins_edges[0]:.0f} kuni {r_bins_edges[1]:.0f} päeva ({days_to_years(r_bins_edges[0])}–{days_to_years(r_bins_edges[1])} aastat tagasi)", "Klientide arv (tk)": r_counts.get(5, 0), "Kirjeldus": "Kõige värskemad kliendid"},
    {"Mõõdik": "Recency (Värskus)", "Skoor": 4, "Vahemik": f"{r_bins_edges[1]:.0f} kuni {r_bins_edges[2]:.0f} päeva ({days_to_years(r_bins_edges[1])}–{days_to_years(r_bins_edges[2])} aastat tagasi)", "Klientide arv (tk)": r_counts.get(4, 0), "Kirjeldus": "Hiline aktiivsus"},
    {"Mõõdik": "Recency (Värskus)", "Skoor": 3, "Vahemik": f"{r_bins_edges[2]:.0f} kuni {r_bins_edges[3]:.0f} päeva ({days_to_years(r_bins_edges[2])}–{days_to_years(r_bins_edges[3])} aastat tagasi)", "Klientide arv (tk)": r_counts.get(3, 0), "Kirjeldus": "Keskmine aktiivsus"},
    {"Mõõdik": "Recency (Värskus)", "Skoor": 2, "Vahemik": f"{r_bins_edges[3]:.0f} kuni {r_bins_edges[4]:.0f} päeva ({days_to_years(r_bins_edges[3])}–{days_to_years(r_bins_edges[4])} aastat tagasi)", "Klientide arv (tk)": r_counts.get(2, 0), "Kirjeldus": "Madal aktiivsus"},
    {"Mõõdik": "Recency (Värskus)", "Skoor": 1, "Vahemik": f"{r_bins_edges[4]:.0f} kuni {r_bins_edges[5]:.0f} päeva ({days_to_years(r_bins_edges[4])}–{days_to_years(r_bins_edges[5])} aastat tagasi)", "Klientide arv (tk)": r_counts.get(1, 0), "Kirjeldus": "Kõige vanemad / passiivsed"},
    
    # --- FREQUENCY (5 kuni 1) ---
    {"Mõõdik": "Frequency (Sagedus)", "Skoor": 5, "Vahemik": "6 ja rohkem ostu", "Klientide arv (tk)": f_counts.get(5, 0), "Kirjeldus": "Väga kõrge sagedusega kliendid"},
    {"Mõõdik": "Frequency (Sagedus)", "Skoor": 4, "Vahemik": "4 või 5 ostu", "Klientide arv (tk)": f_counts.get(4, 0), "Kirjeldus": "Aktiivsed püsikliendid"},
    {"Mõõdik": "Frequency (Sagedus)", "Skoor": 3, "Vahemik": "Täpselt 3 ostu", "Klientide arv (tk)": f_counts.get(3, 0), "Kirjeldus": "Regulaarsed ostjad"},
    {"Mõõdik": "Frequency (Sagedus)", "Skoor": 2, "Vahemik": "Täpselt 2 ostu", "Klientide arv (tk)": f_counts.get(2, 0), "Kirjeldus": "Korduvostjad"},
    {"Mõõdik": "Frequency (Sagedus)", "Skoor": 1, "Vahemik": "Täpselt 1 ost", "Klientide arv (tk)": f_counts.get(1, 0), "Kirjeldus": "Ühekordsed ostjad"},

    # --- MONETARY (5 kuni 1) ---
    {"Mõõdik": "Monetary (Käive)", "Skoor": 5, "Vahemik": f"{round(m_bins_edges[4])} € kuni {round(m_bins_edges[5])} €", "Klientide arv (tk)": m_counts.get(5, 0), "Kirjeldus": "Kõige suuremad summad"},
    {"Mõõdik": "Monetary (Käive)", "Skoor": 4, "Vahemik": f"{round(m_bins_edges[3])} € kuni {round(m_bins_edges[4])} €", "Klientide arv (tk)": m_counts.get(4, 0), "Kirjeldus": "Kõrgemad summad"},
    {"Mõõdik": "Monetary (Käive)", "Skoor": 3, "Vahemik": f"{round(m_bins_edges[2])} € kuni {round(m_bins_edges[3])} €", "Klientide arv (tk)": m_counts.get(3, 0), "Kirjeldus": "Keskmised summad"},
    {"Mõõdik": "Monetary (Käive)", "Skoor": 2, "Vahemik": f"{round(m_bins_edges[1])} € kuni {round(m_bins_edges[2])} €", "Klientide arv (tk)": m_counts.get(2, 0), "Kirjeldus": "Keskmisest madalamad summad"},
    {"Mõõdik": "Monetary (Käive)", "Skoor": 1, "Vahemik": f"{round(m_bins_edges[0])} € kuni {round(m_bins_edges[1])} €", "Klientide arv (tk)": m_counts.get(1, 0), "Kirjeldus": "Madalamad summad"}
]

# Loome DataFrame'i ja kuvame
rfm_criteria_table = pd.DataFrame(criteria_list)
display(rfm_criteria_table)

# (Valikuline) Salvestamine Excelisse:
# rfm_criteria_table.to_excel('rfm_criteria_sorted_5_to_1.xlsx', index=False)

,Mõõdik,Skoor,Vahemik,Klientide arv (tk),Kirjeldus
0,Recency (Värskus),5,1 kuni 174 päeva (0.0–0.5 aastat tagasi),646,Kõige värskemad kliendid
1,Recency (Värskus),4,174 kuni 304 päeva (0.5–0.8 aastat tagasi),655,Hiline aktiivsus
2,Recency (Värskus),3,304 kuni 683 päeva (0.8–1.9 aastat tagasi),639,Keskmine aktiivsus
3,Recency (Värskus),2,683 kuni 1218 päeva (1.9–3.3 aastat tagasi),636,Madal aktiivsus
4,Recency (Värskus),1,1218 kuni 2103 päeva (3.3–5.8 aastat tagasi),643,Kõige vanemad / passiivsed
5,Frequency (Sagedus),5,6 ja rohkem ostu,18,Väga kõrge sagedusega kliendid
6,Frequency (Sagedus),4,4 või 5 ostu,52,Aktiivsed püsikliendid
7,Frequency (Sagedus),3,Täpselt 3 ostu,83,Regulaarsed ostjad
8,Frequency (Sagedus),2,Täpselt 2 ostu,345,Korduvostjad
9,Frequency (Sagedus),1,Täpselt 1 ost,2721,Ühekordsed ostjad
